In [4]:
import dimcli
import pandas as pd
import duckdb
import json

In [5]:
dimcli.login()
dsl = dimcli.Dsl()

Searching config file credentials for default 'live' instance..


Dimcli - Dimensions API Client (v1.7)
Connected to: <https://app.dimensions.ai/api/dsl> - DSL v2.15
Method: dsl.ini file
Couldn't connect to the pypi server. Are you online?


In [72]:
training_data = pd.read_csv("patents_curated.csv")

In [97]:
ids = training_data['id'].tolist()
family_ids = training_data['family_id'].dropna().unique().astype(int).tolist()

In [83]:
def chunks(list, n):
    for i in range(0, len(list), n):
        yield list[i:i + n]

In [33]:
for i in range(0, len(ids), 500): print(i)

0
500
1000
1500
2000


In [121]:
## look for all patents with the respective ids
query = []
for i in range(0, len(ids), 500):
    ids_batch = ids[i:i+500]
    q = dsl.query(f"""search patents
          where id in {json.dumps(ids_batch)}
          return patents[id+family_id+application_number+title+abstract+cpc+jurisdiction+
                        publication_date+publication_year+priority_year+filing_date+filing_status+original_assignees]
          limit 500""")
    query.append(q)
    

Returned Patents: 500 (total = 500)
Time: 2.24s
Returned Patents: 500 (total = 500)
Time: 5.78s
Returned Patents: 500 (total = 500)
Time: 5.90s
Returned Patents: 500 (total = 500)
Time: 6.37s
Returned Patents: 329 (total = 329)
Time: 2.09s


In [122]:
## look for all patents with the respective family ids
for i in range(0, len(family_ids), 500):
    ids_batch = family_ids[i:i+500]
    q = dsl.query_iterative(f"""search patents
          where family_id in {json.dumps(ids_batch)}
          return patents[id+family_id+application_number+title+abstract+cpc+jurisdiction+
                        publication_date+publication_year+priority_year+filing_date+filing_status+original_assignees] 
          """)
    query.append(q)

Starting iteration with limit=1000 skip=0 ...
0-1000 / 5710 (2.35s)
1000-2000 / 5710 (2.36s)
2000-3000 / 5710 (3.81s)
3000-4000 / 5710 (2.18s)
4000-5000 / 5710 (2.41s)
5000-5710 / 5710 (3.30s)
===
Records extracted: 5710
Starting iteration with limit=1000 skip=0 ...
0-1000 / 2076 (6.11s)
1000-2000 / 2076 (2.56s)
2000-2076 / 2076 (1.92s)
===
Records extracted: 2076
Starting iteration with limit=1000 skip=0 ...
0-958 / 958 (2.42s)
===
Records extracted: 958


In [ ]:
# join all query results together
df = pd.concat([q.as_dataframe() for q in query], ignore_index=True)

In [ ]:
# deduplicate by id
df = df.drop_duplicates(subset="id").reset_index(drop=True)

In [ ]:
# remove <..> in abstracts
df['abstract'] = df['abstract'].str.replace(r'<[^>]*>', '', regex=True)

In [ ]:
# check how many NAs in abstracts
any(df["abstract"].isna())
df[df['abstract'].isna()]

,id,title,abstract,application_number,cpc,filing_date,filing_status,jurisdiction,original_assignees,priority_year,publication_date,publication_year
119,RS-67291-B1,COLLAGEN HYDROGELS USEFUL AS CELL CARRIERS,NaN,RS20250969,"[A61L27/24, A23L13/00, C12N2533/54, C12N5/0068...",2022-04-19,N/A,RS,"[{'city_name': 'Cáseda', 'country_code': 'ES',...",2021.0,2025-11-28,2025
120,RS-67239-B1,VEGETARIAN BURGER,NaN,RS20250952,"[A23J3/16, A23J3/14, A23J3/18, A23J3/24, A23L5...",2020-10-20,N/A,RS,"[{'country_code': 'NL', 'country_name': 'Nethe...",2019.0,2025-10-31,2025
121,RS-66799-B1,MEAT ANALOGUE COMPRISING AQUEOUS GELLING COMPO...,NaN,RS20250478,"[A23J3/227, A23V2002/00, A23J3/18, A23J3/16, A...",2018-03-08,N/A,RS,"[{'country_code': 'NL', 'country_name': 'Nethe...",2017.0,2025-06-30,2025
122,RS-66499-B1,MEAT ANALOGUE AND PROCESS FOR PRODUCING THE SAME,NaN,RS20250146,"[A23J3/14, A23L29/035, A23J3/227, A23J3/16]",2021-12-29,N/A,RS,"[{'country_code': 'NL', 'country_name': 'Nethe...",2020.0,2025-03-31,2025
145,JP-7791811-B2,Composition for producing test soils for evalu...,NaN,JP2022501220,"[A61B90/98, B08B2209/08, A61B2090/702, B08B9/4...",2020-07-08,Grant,JP,"[{'city_name': 'Offenburg', 'country_code': 'D...",2019.0,2025-12-24,2025
...,...,...,...,...,...,...,...,...,...,...,...,...
8806,CA-3233687-A1,SOYBEAN VARIETY,NaN,CA3233687,"[A01H6/542, A01H5/10]",2024-03-28,Application,CA,"[{'city_name': 'Basel', 'country_code': 'CH', ...",2024.0,2025-10-30,2025
8807,CA-3233203-A1,SOYBEAN VARIETY,NaN,CA3233203,"[A01H6/542, A01H5/10]",2024-03-25,Application,CA,"[{'city_name': 'Basel', 'country_code': 'CH', ...",2024.0,2025-07-08,2025
8808,CA-3232136-A1,SOYBEAN VARIETY,NaN,CA3232136,"[A01H6/542, A01H5/10]",2024-03-15,Application,CA,"[{'city_name': 'Basel', 'country_code': 'CH', ...",2024.0,2025-07-08,2025
8813,AU-2024395542-A1,SULFONAMIDE DERIVATIVES AND USE THEREOF IN THE...,NaN,AU2024395542,"[C07D413/14, C07D401/14, C07D211/22, C07D401/1...",2024-12-06,Application,AU,"[{'city_name': 'Tours', 'country_code': 'FR', ...",2023.0,2026-05-28,2026


In [156]:
# function to select 1-2 patent per family
def select_patents(group):
    
    preferred_jurisdictions = ['WO', 'EP', 'US']  # note: WO not WP?
    
    # Split into has content / no content
    has_content = group[
        group['title'].notna() & group['abstract'].notna() &
        (group['title'] != '') & (group['abstract'] != '')
    ].copy()
    
    # If nothing has title+abstract, just return newest
    if has_content.empty:
        return group.sort_values('publication_date', ascending=False).head(1)
    
    # Sort by preferred jurisdiction, then newest date
    has_content['_preferred'] = has_content['jurisdiction'].isin(preferred_jurisdictions)
    has_content = has_content.sort_values(
        ['_preferred', 'publication_date'], ascending=[False, False]
    )
    
    # Deduplicate by unique content — keeps best (preferred jurisdiction, newest) per unique text
    unique_content = has_content.drop_duplicates(subset=['title', 'abstract'])
    
    return unique_content.head(2).drop(columns='_preferred')

In [ ]:
# get selected patents (reset index part needed to keep family_id)
patents_for_training = df.groupby('family_id', group_keys=True).apply(select_patents).reset_index(level=0).reset_index(drop=True)

In [160]:
patents_for_training

,family_id,id,title,abstract,application_number,cpc,filing_date,filing_status,jurisdiction,original_assignees,priority_year,publication_date,publication_year
0,38969498.0,US-11540543-B2,Sweetened consumables comprising mogroside IV ...,Disclosed are sweetened consumables and method...,US17036785,"[A23L29/37, A23L29/30, A23L33/105, A23L21/00, ...",2020-09-29,Grant,US,"[{'city_name': 'Vernier', 'country_code': 'CH'...",2006.0,2023-01-03,2023
1,38969498.0,US-20210015134-A1,CONSUMABLES,Disclosed are sweetened consumables and method...,US17036785,"[A23L27/30, A23L2/60, A23V2002/00, A23L27/33, ...",2020-09-29,Application,US,"[{'city_name': 'Vernier', 'country_code': 'CH'...",2006.0,2021-01-21,2021
2,42829610.0,US-20170298457-A1,LACTIC BACTERIUM WITH MODIFIED GALACTOKINASE E...,The present invention relates to a bacterial c...,US15360298,"[A23C19/0323, C12N9/1205, A23C2220/206, A23V24...",2016-11-23,Application,US,"[{'city_name': 'Horsholm', 'country_code': 'DK...",2009.0,2017-10-19,2017
3,42829610.0,EP-2473058-A1,LACTIC BACTERIUM WITH MODIFIED GALACTOKINASE E...,The present invention relates to a bacterial c...,EP10751643A,"[A23C19/0323, C12R2001/46, C12Y207/01006, A23L...",2010-09-01,Application,EP,"[{'city_name': 'Horsholm', 'country_code': 'DK...",2009.0,2012-07-11,2012
4,44510082.0,EP-3542638-A1,NUTRITIONAL COMPOSITION,Non-medical use of at least two components sel...,EP19173302.1,"[A23L33/13, A61K33/04, A23L33/40, A61K31/14, A...",2011-08-11,Application,EP,"[{'city_name': 'HM ZOETERMEER', 'country_code'...",2010.0,2019-09-25,2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2454,99798170.0,WO-2025079037-A2,"PRODUCT, SYSTEM AND METHOD OF CELL CULTIVATION",The present invention relates to a cell biomas...,IB2024/059990,"[C12P21/02, C12N2500/38, C12N2500/32, C12P21/0...",2024-10-11,Application,WO,NaN,2023.0,2025-04-17,2025
2455,99799704.0,WO-2025113823-A1,POWDER MIX FORMULATIONS WITH IMPROVED SENSORY ...,The invention relates to a powder preparation ...,EP2024/025329,"[A23L2/66, A23L2/39, A23L33/185, A23L33/105, A...",2024-11-26,Application,WO,"[{'city_name': 'Lestrem', 'country_code': 'FR'...",2023.0,2025-06-05,2025
2456,99799723.0,US-20250236833-A1,"Product, system and method of cell cultivation","The present invention provides products, syste...",US19028830,"[C12N5/0653, A23K40/20, A23L33/10, A23K40/25, ...",2025-01-17,Application,US,NaN,2023.0,2025-07-24,2025
2457,99799739.0,US-20250270503-A1,"Product, system and method of cell cultivation","The present invention provides products, syste...",US19028930,"[A23K10/20, C12N2510/04, C12M33/10, C12N2510/0...",2025-01-17,Application,US,NaN,2023.0,2025-08-28,2025


In [183]:
# join with scope and pillar/category/etc information from training data

# Step 1: filter
mask = (patents_for_training['id'].isin(training_data['id']) | 
        patents_for_training['family_id'].isin(training_data['family_id']))
result = patents_for_training[mask].copy()

# Step 2a: merge by exact id
result = result.merge(training_data[['id', 'scope', 'pillar', 'research_category', 'endproduct', 'ingredient', 'subpillar']], 
                      on='id', how='left')

# Step 2b: merge by family_id as fallback
family_lookup = training_data.drop_duplicates('family_id')[['family_id', 'scope', 'pillar', 'research_category', 'endproduct','ingredient', 'subpillar']]
result = result.merge(family_lookup, on='family_id', how='left', suffixes=('', '_family'))

# Fill gaps from id-merge with family-merge results
result['scope'] = result['scope'].fillna(result['scope_family'])
result['pillar'] = result['pillar'].fillna(result['pillar_family'])
result['research_category'] = result['research_category'].fillna(result['research_category_family'])
result['endproduct'] = result['endproduct'].fillna(result['endproduct_family'])
result['ingredient'] = result['ingredient'].fillna(result['ingredient_family'])
result['subpillar'] = result['subpillar'].fillna(result['subpillar_family'])
result = result.drop(columns=['scope_family', 'pillar_family', 'research_category_family', 'endproduct_family', 'ingredient_family', 'subpillar_family'])

In [184]:
result


,family_id,id,title,abstract,application_number,cpc,filing_date,filing_status,jurisdiction,original_assignees,priority_year,publication_date,publication_year,scope,pillar,research_category,endproduct,ingredient,subpillar
0,38969498.0,US-11540543-B2,Sweetened consumables comprising mogroside IV ...,Disclosed are sweetened consumables and method...,US17036785,"[A23L29/37, A23L29/30, A23L33/105, A23L21/00, ...",2020-09-29,Grant,US,"[{'city_name': 'Vernier', 'country_code': 'CH'...",2006.0,2023-01-03,2023,out,NaN,NaN,NaN,NaN,NaN
1,38969498.0,US-20210015134-A1,CONSUMABLES,Disclosed are sweetened consumables and method...,US17036785,"[A23L27/30, A23L2/60, A23V2002/00, A23L27/33, ...",2020-09-29,Application,US,"[{'city_name': 'Vernier', 'country_code': 'CH'...",2006.0,2021-01-21,2021,out,NaN,NaN,NaN,NaN,NaN
2,42829610.0,US-20170298457-A1,LACTIC BACTERIUM WITH MODIFIED GALACTOKINASE E...,The present invention relates to a bacterial c...,US15360298,"[A23C19/0323, C12N9/1205, A23C2220/206, A23V24...",2016-11-23,Application,US,"[{'city_name': 'Horsholm', 'country_code': 'DK...",2009.0,2017-10-19,2017,in,PB,Strain development,Yoghurt and fermented dairy,NaN,NaN
3,42829610.0,EP-2473058-A1,LACTIC BACTERIUM WITH MODIFIED GALACTOKINASE E...,The present invention relates to a bacterial c...,EP10751643A,"[A23C19/0323, C12R2001/46, C12Y207/01006, A23L...",2010-09-01,Application,EP,"[{'city_name': 'Horsholm', 'country_code': 'DK...",2009.0,2012-07-11,2012,in,PB,Strain development,Yoghurt and fermented dairy,NaN,NaN
4,44510082.0,EP-3542638-A1,NUTRITIONAL COMPOSITION,Non-medical use of at least two components sel...,EP19173302.1,"[A23L33/13, A61K33/04, A23L33/40, A61K31/14, A...",2011-08-11,Application,EP,"[{'city_name': 'HM ZOETERMEER', 'country_code'...",2010.0,2019-09-25,2019,out,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2454,99798170.0,WO-2025079037-A2,"PRODUCT, SYSTEM AND METHOD OF CELL CULTIVATION",The present invention relates to a cell biomas...,IB2024/059990,"[C12P21/02, C12N2500/38, C12N2500/32, C12P21/0...",2024-10-11,Application,WO,NaN,2023.0,2025-04-17,2025,in,CM,Bioprocess design,Meat,NaN,NaN
2455,99799704.0,WO-2025113823-A1,POWDER MIX FORMULATIONS WITH IMPROVED SENSORY ...,The invention relates to a powder preparation ...,EP2024/025329,"[A23L2/66, A23L2/39, A23L33/185, A23L33/105, A...",2024-11-26,Application,WO,"[{'city_name': 'Lestrem', 'country_code': 'FR'...",2023.0,2025-06-05,2025,in,PB,Ingredient optimisation,Cross-cutting,"Isolates, concentates, and flours",NaN
2456,99799723.0,US-20250236833-A1,"Product, system and method of cell cultivation","The present invention provides products, syste...",US19028830,"[C12N5/0653, A23K40/20, A23L33/10, A23K40/25, ...",2025-01-17,Application,US,NaN,2023.0,2025-07-24,2025,in,CM,Bioprocess design,Meat,NaN,NaN
2457,99799739.0,US-20250270503-A1,"Product, system and method of cell cultivation","The present invention provides products, syste...",US19028930,"[A23K10/20, C12N2510/04, C12M33/10, C12N2510/0...",2025-01-17,Application,US,NaN,2023.0,2025-08-28,2025,in,CM,Bioprocess design,Meat,NaN,NaN


In [185]:
result.to_csv('patents_training_data.csv')

In [6]:
result = pd.read_csv('patents_training_data.csv')

In [7]:
duckdb.connect("../patents_training.db")

### Retrieve the CPC codes that cover all entries for a first filter after the dimensions query

In [ ]:
# Get codes that cover all entries with a minimum number of codes
rows = df['cpc'].dropna().tolist()
uncovered = set(range(len(rows)))
selected_codes = []

while uncovered:
    # Find the code that covers the most uncovered rows
    code_coverage = {}
    for idx in uncovered:
        for code in rows[idx]:
            code_coverage[code] = code_coverage.get(code, set()) | {idx}
    
    best_code = max(code_coverage, key=lambda c: len(code_coverage[c]))
    selected_codes.append(best_code)
    uncovered -= code_coverage[best_code]

print(f"Cover size: {len(selected_codes)}")
print(selected_codes)

Cover size: 257
['A23J3/227', 'A23V2002/00', 'A23J3/14', 'A23L33/40', 'A23L13/00', 'A23C11/10', 'A01H5/10', 'A23L27/88', 'C12N1/205', 'A23L33/21', 'A61Q19/00', 'A23J3/20', 'A23K50/40', 'A47J31/4489', 'A47J31/4485', 'A23C11/103', 'A23C20/02', 'A23G1/48', 'C12N2513/00', 'A23L33/185', 'A23K10/30', 'C12N2510/00', 'A23L29/256', 'A23C11/02', 'A23P30/20', 'A61K8/37', 'A23L33/135', 'A23L2/39', 'C12M21/08', 'C07K14/415', 'C12N1/14', 'A23L33/115', 'A23B2/754', 'C12N15/52', 'A23K50/10', 'B65D65/466', 'A61Q13/00', 'A22C25/16', 'C12N9/80', 'C12P19/04', 'A23G9/327', 'A47J31/4403', 'A23L33/105', 'A23K20/147', 'A23C9/1522', 'A23P20/20', 'A61Q11/00', 'A61P1/00', 'C11B1/10', 'C12M29/04', 'C12M23/26', 'G06N3/09', 'A23F5/405', 'G01N33/04', 'A23C20/00', 'A61P19/00', 'A23L19/09', 'C07K2319/02', 'A23C11/06', 'A61P35/00', 'C12M31/10', 'A61P3/10', 'B08B13/00', 'A61K39/39', 'A23L2/52', 'A47J37/0664', 'C11B9/00', 'A23L11/05', 'C12N15/81', 'B26D2210/02', 'C07K14/503', 'C12N5/0658', 'A23P20/11', 'B07B1/46', 'C12N1

In [ ]:
# check whether all entries have been covered
covered = df['cpc'].dropna().apply(lambda codes: any(c in selected_codes for c in codes))
assert covered.all()

In [71]:
# save codes
with open('../CPC_for_filter.txt', 'w') as f:
    f.write('\n'.join(selected_codes))

# Read it back in as list
# with open('selected_codes.txt', 'r') as f:
#    selected_codes = f.read().splitlines()

### CPC codes for initial search

In [2]:
cpc_codes = [
    "A23C11/065", "A23C11/10", "A23C11/103", "A23C11/106",
    "A23C20/005", "A23C20/02", "A23C20/025",
    "A23J1/005", "A23J1/006", "A23J1/007", "A23J1/008", "A23J1/009",
    "A23J1/12", "A23J1/14", "A23J1/18",
    "A23J3/14", "A23J3/16", "A23J3/18", "A23J3/20", "A23J3/225", "A23J3/227",
    "A23L11/40", "A23L11/45", "A23L11/50", "A23L11/60", "A23L11/65",
    "A23L15/35", "A23L17/35", "A23L31/00", "A23L33/185", "A23L33/195",
    "A23V2200/264",
    "C12N5/0043", "C12N5/0653", "C12N5/0658"
]

In [3]:
# save codes
with open('../CPC_for_query.txt', 'w') as f:
    f.write('\n'.join(cpc_codes))